<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/Browser_History_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To analyze a simulated browser-history dataset using Python by identifying frequently visited domains, browsing activity within a specified time period, and URLs matching predefined suspicious patterns, while presenting normal and potentially suspicious activities separately.

**Algorithm**

Create or load the simulated browser-history dataset.

Read the URL, timestamp, domain, and visit-count fields.

Convert timestamps into datetime format.

Count and rank domains according to their visit frequency.

Select browsing records within the specified time period.

Define suspicious URL patterns such as phishing-related keywords, IP-based URLs, and suspicious download extensions.

Scan URLs against the predefined patterns.

Separate normal high-frequency browsing from potentially suspicious activity.

Display the domain-frequency summary.

Display the suspicious URL report with timestamps and domains.

In [1]:
# ==============================================
# BROWSER HISTORY FORENSIC ANALYZER
# ==============================================

import pandas as pd
import re

# ------------------------------------------------
# 1. Simulated Browser History
# ------------------------------------------------

data = [
    ["2026-08-25 08:10:00", "https://www.google.com/search?q=python",
     "google.com", 15],

    ["2026-08-25 08:20:00", "https://www.youtube.com/watch?v=123",
     "youtube.com", 12],

    ["2026-08-25 08:35:00", "https://github.com/python",
     "github.com", 10],

    ["2026-08-25 09:00:00", "https://www.google.com/search?q=cybersecurity",
     "google.com", 8],

    ["2026-08-25 09:15:00", "https://www.youtube.com/security",
     "youtube.com", 9],

    ["2026-08-25 09:30:00", "http://192.168.1.50/login",
     "192.168.1.50", 2],

    ["2026-08-25 09:40:00", "http://fake-login.com/verify",
     "fake-login.com", 3],

    ["2026-08-25 09:50:00",
     "https://example.com/free-download.exe",
     "example.com", 1],

    ["2026-08-25 10:05:00",
     "https://www.google.com/search?q=linux",
     "google.com", 6],

    ["2026-08-25 10:20:00",
     "https://github.com/security/tools",
     "github.com", 7]
]

df = pd.DataFrame(
    data,
    columns=[
        "Timestamp",
        "URL",
        "Domain",
        "Visit_Count"
    ]
)

# Convert timestamp
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"]
)

# ------------------------------------------------
# 2. Configurable Time Period
# ------------------------------------------------

START_TIME = "2026-08-25 08:00:00"
END_TIME   = "2026-08-25 10:00:00"

# ------------------------------------------------
# 3. Most Frequently Visited Domains
# ------------------------------------------------

domain_summary = (
    df.groupby("Domain")["Visit_Count"]
      .sum()
      .sort_values(ascending=False)
)

# ------------------------------------------------
# 4. Browsing Activity in Time Period
# ------------------------------------------------

period_activity = df[
    (df["Timestamp"] >= START_TIME) &
    (df["Timestamp"] <= END_TIME)
].copy()

# ------------------------------------------------
# 5. Suspicious URL Patterns
# ------------------------------------------------

suspicious_patterns = {

    "IP address used as URL":
        r"https?://\d{1,3}(\.\d{1,3}){3}",

    "Login/verification keyword":
        r"(login|verify|verification|account)",

    "Executable download":
        r"\.(exe|scr|bat|cmd)(\?|$)",

    "Suspicious download keyword":
        r"(free-download|crack|keygen|payload)"
}

# ------------------------------------------------
# 6. Scan URLs
# ------------------------------------------------

suspicious_records = []

for _, row in df.iterrows():

    reasons = []

    url = row["URL"].lower()

    for reason, pattern in suspicious_patterns.items():

        if re.search(pattern, url):

            reasons.append(reason)

    if reasons:

        suspicious_records.append({
            "Timestamp": row["Timestamp"],
            "URL": row["URL"],
            "Domain": row["Domain"],
            "Visit_Count": row["Visit_Count"],
            "Reason": "; ".join(reasons)
        })

suspicious_df = pd.DataFrame(
    suspicious_records
)

# ------------------------------------------------
# 7. Normal High-Frequency Browsing
# ------------------------------------------------

HIGH_FREQUENCY_THRESHOLD = 10

normal_high_frequency = df[
    df["Visit_Count"] >= HIGH_FREQUENCY_THRESHOLD
].copy()

# Remove suspicious URLs from normal summary
if not suspicious_df.empty:

    normal_high_frequency = normal_high_frequency[
        ~normal_high_frequency["URL"].isin(
            suspicious_df["URL"]
        )
    ]

# ------------------------------------------------
# 8. Display Report
# ------------------------------------------------

print("=" * 90)
print("                 BROWSER HISTORY FORENSIC ANALYSIS")
print("=" * 90)

# -----------------------------------------------
# Frequent Domains
# -----------------------------------------------

print("\n" + "-" * 90)
print("MOST FREQUENTLY VISITED DOMAINS")
print("-" * 90)

print(
    domain_summary.to_string()
)

# -----------------------------------------------
# Time Period Activity
# -----------------------------------------------

print("\n" + "-" * 90)
print("BROWSING ACTIVITY WITHIN SELECTED PERIOD")
print("-" * 90)

print(
    f"Period: {START_TIME} to {END_TIME}\n"
)

print(
    period_activity.to_string(index=False)
)

# -----------------------------------------------
# Normal High Frequency
# -----------------------------------------------

print("\n" + "-" * 90)
print("NORMAL HIGH-FREQUENCY BROWSING")
print("-" * 90)

if not normal_high_frequency.empty:

    print(
        normal_high_frequency[
            ["Domain", "URL", "Visit_Count"]
        ].to_string(index=False)
    )

else:

    print("No high-frequency normal browsing detected.")

# -----------------------------------------------
# Suspicious Activity
# -----------------------------------------------

print("\n" + "-" * 90)
print("POTENTIALLY SUSPICIOUS ACTIVITY")
print("-" * 90)

if not suspicious_df.empty:

    print(
        suspicious_df.to_string(index=False)
    )

else:

    print("No predefined suspicious patterns detected.")

print("\n" + "=" * 90)
print("Browser history analysis completed.")
print("=" * 90)

                 BROWSER HISTORY FORENSIC ANALYSIS

------------------------------------------------------------------------------------------
MOST FREQUENTLY VISITED DOMAINS
------------------------------------------------------------------------------------------
Domain
google.com        29
youtube.com       21
github.com        17
fake-login.com     3
192.168.1.50       2
example.com        1

------------------------------------------------------------------------------------------
BROWSING ACTIVITY WITHIN SELECTED PERIOD
------------------------------------------------------------------------------------------
Period: 2026-08-25 08:00:00 to 2026-08-25 10:00:00

          Timestamp                                           URL         Domain  Visit_Count
2026-08-25 08:10:00        https://www.google.com/search?q=python     google.com           15
2026-08-25 08:20:00           https://www.youtube.com/watch?v=123    youtube.com           12
2026-08-25 08:35:00                     htt

**Result**

The Python browser-history analyzer successfully examined the simulated browsing records and identified the most frequently visited domains, browsing activity within the specified time period, and URLs matching predefined suspicious patterns. Normal high-frequency browsing was presented separately from potentially suspicious activity. The tool therefore helps an investigator quickly distinguish routine browsing behavior from URLs requiring further forensic examination, without automatically declaring a flagged URL malicious.